# Quadrature Time: interactive notebook

Companion to `paper/quadrature_time.pdf`. Every quantity here comes from an **explicit bipartite state** |Ψ⟩ ∈ ℂᴺ ⊗ ℂ² and real partial traces against clock POVMs.

1. Exact Page–Wootters conditioning: uniform marginal, exact Schrödinger evolution (Theorem 1)
2. Unsharp readout: a *static* suppression of coherence (Proposition 5)
3. Diffusive clock error: Lindblad decay, under Assumption 1 (Proposition 6)
4. Winding marginalization on a cyclic clock (Proposition 2)
5. Relativistic clock interferometer with revivals


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import numpy as np, matplotlib.pyplot as plt
from qtime import clocks as qc, interferometer as ifm
try:
    from ipywidgets import interact, IntSlider, FloatSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
print('widgets available:', HAVE_WIDGETS)

## 1–2. Exact conditioning, sharp vs. unsharp readout
The system gap `r` must be an **integer** number of clock quanta: on a circle clock the constraint has no solution otherwise (Sec. II). Try `sigma = 0`, which gives exact unitary evolution.

In [ ]:
def conditioning(N=128, r=4, sigma=0.15):
    psi = qc.pw_constraint_state(N, [0, r], [1, 1])
    th = qc.phase_grid(N)
    p_s, rho_s = qc.condition_on_clock(psi, qc.sharp_phase_povm(N))
    p_u, rho_u = qc.condition_on_clock(psi, qc.smeared_phase_povm(N, sigma))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(th, N * p_s, 'k', label='sharp'); ax[0].plot(th, N * p_u, 'C1--', label='unsharp')
    ax[0].set_ylim(0, 1.5); ax[0].set_title('raw clock marginal  N p(θ)'); ax[0].legend()
    ax[1].plot(th, rho_s[:, 0, 1].real, 'k', lw=1, label='sharp Re ρ01')
    ax[1].plot(th, rho_u[:, 0, 1].real, 'C1', lw=2, label='unsharp Re ρ01')
    a = 0.5 * qc.coherence_suppression_static(r, sigma)
    ax[1].axhline(a, color='C2', ls=':'); ax[1].axhline(-a, color='C2', ls=':')
    ax[1].set_title(f'|ρ01| unsharp = {abs(rho_u[0,0,1]):.4f}, analytic = {a:.4f}'); ax[1].legend()
    plt.show()

if HAVE_WIDGETS:
    interact(conditioning, N=IntSlider(min=32, max=256, step=32, value=128),
             r=IntSlider(min=1, max=12, value=4), sigma=FloatSlider(min=0, max=0.6, step=0.025, value=0.15))
else:
    conditioning()

## 3. Diffusive clock: where the Lindblad equation comes from
The decay appears only if the clock's timing variance grows, σ_t² = T_C t. Three independent computations agree: direct quadrature of the smearing channel, the closed form, and numerical GKSL integration.

In [ ]:
E = np.array([0.0, 4.0]); rho0 = 0.5 * np.ones((2, 2)); t = np.linspace(0, 2*np.pi, 60); T_C = 0.01
quad = qc.diffusive_clock_channel(rho0, E, t, T_C, n_quad=401)
closed = qc.lindblad_dephasing_solution(rho0, E, t, T_C)
ode = qc.integrate_lindblad(rho0, np.diag(E), t, T_C)
plt.plot(t, abs(quad[:, 0, 1]), 'o', ms=3, label='quadrature'); plt.plot(t, abs(closed[:, 0, 1]), label='closed form')
plt.plot(t, abs(ode[:, 0, 1]), '--', label='GKSL ODE'); plt.xlabel('ωt'); plt.ylabel('|ρ01|'); plt.legend(); plt.show()
print('max deviation:', np.abs(quad - closed).max(), np.abs(ode - closed).max())

## 4. Winding marginalization
A cyclic clock read without counting its turns keeps coherence only for integer ν = (E_j − E_k)/ħω.

In [ ]:
def winding(M=10):
    nu = np.linspace(-0.25, 3.25, 2001)
    plt.plot(nu, [abs(qc.winding_marginal_coherence(x, M)) for x in nu])
    plt.xlabel('ν = (E_j − E_k)/ħω'); plt.ylabel('surviving coherence'); plt.title(f'M = {M} uncounted windings'); plt.show()
if HAVE_WIDGETS:
    interact(winding, M=IntSlider(min=1, max=80, value=10))
else:
    winding()

## 5. Relativistic clock interferometer
The internal spectrum is equally spaced, so the visibility **revives** exactly every T_rev = 2π/(ω₀|Δ|).

In [ ]:
def interferometer(p1=0.2, p2=2.5, omega0=8.0, dn=6.0, revivals=4):
    d = 1/ifm.lorentz_gamma(p1) - 1/ifm.lorentz_gamma(p2)
    T = ifm.revival_period(omega0, d); t = np.arange(0, revivals*T + T/5, ifm.suggested_time_step(dn, omega0, d))
    pops = ifm.gaussian_populations(60, dn)
    plt.figure(figsize=(10, 3.5)); plt.plot(t, ifm.visibility(t, pops, omega0, d), label='V(t)')
    plt.plot(t, ifm.gaussian_envelope(t, dn, omega0, d), 'r--', label='short-time envelope')
    plt.title(f'T_rev = {T:.3f}'); plt.xlabel('t'); plt.legend(); plt.show()
if HAVE_WIDGETS:
    interact(interferometer, p1=(0.0, 3.0, 0.1), p2=(0.0, 3.0, 0.1), omega0=(1.0, 20.0, 1.0), dn=(1.0, 12.0, 0.5), revivals=(1, 8))
else:
    interferometer()